# SSJ on a GPU — does the ratio move?

On CPU, Simultaneous Saturated Jacobi is **24–46× slower than LAPACK
`dsyevd`**. That is not an implementation gap: SSJ costs ~2.67
gemm-equivalents per sweep and needs 20–29 sweeps, so ~55–80
gemm-equivalents, against `dsyevd`'s measured 8–18. It loses on flops.

The reason to look at a GPU is that SSJ's flops are *all gemm* (with
`method="gemm"`, every operation is a matmul), while `syevd` spends most of
its time in tridiagonal reduction — half of that in memory-bound `symv`,
which GPUs execute badly. So the honest question is not "is SSJ fast?" but:

> **Does the SSJ-to-incumbent ratio shrink when moving from CPU+LAPACK to
> GPU+cuSOLVER — and by how much?**

This notebook measures exactly that. It runs the same solver on the GPU and
on the CPU in one session and reports both ratios, so the answer is a
*difference of ratios* and cannot be confounded by which machine Colab gave
you.

**Before you run anything: Runtime → Change runtime type → GPU.**

### The caveat that decides how to read every number below

Consumer GPUs are deliberately crippled in float64. A T4 or P100 runs fp64 at
**1/32** of its fp32 rate; an A100 or V100 runs it at **1/2**. Eigensolvers
are normally fp64. Cell 1 *measures* your GPU's ratio rather than assuming it,
and prints which rows of the results table are the meaningful ones for the
card you actually got.

*The solver code in cell 2 is a copy of `src/ssj/core.py` from the
`general-eigensolver` repository, kept standalone so the notebook runs without
repository access. Two deliberate deviations, both GPU-specific and both
marked in the source: the power iterations avoid per-iteration host syncs, and
the block pass in cell 3 is batched.*

In [ ]:
# =====================================================================
#  1 - GPU probe, and the fp64/fp32 ratio that frames everything below
# =====================================================================
import subprocess, sys, time

smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
     "--format=csv,noheader"],
    capture_output=True, text=True).stdout.strip()
if not smi:
    raise SystemExit("No GPU visible. Runtime > Change runtime type > GPU, "
                     "then run this cell again.")
print("GPU:", smi)

import cupy as cp
import numpy as np
print("cupy", cp.__version__, "| numpy", np.__version__)


def gemm_gflops(dtype, n=2048, reps=5):
    """Sustained matmul throughput. cp.ones avoids paying for RNG."""
    A = cp.ones((n, n), dtype=dtype)
    B = cp.ones((n, n), dtype=dtype)
    A @ B                                    # warm up: autotune + workspace
    cp.cuda.Device().synchronize()
    t0 = time.perf_counter()
    for _ in range(reps):
        A @ B
    cp.cuda.Device().synchronize()
    dt = time.perf_counter() - t0
    return reps * 2.0 * n ** 3 / dt / 1e9


g32 = gemm_gflops(cp.float32)
g64 = gemm_gflops(cp.float64)
ratio = g32 / g64
print(f"\nfp32 gemm : {g32:8.0f} GFLOP/s")
print(f"fp64 gemm : {g64:8.0f} GFLOP/s")
print(f"fp32:fp64 = {ratio:.1f} : 1")

if ratio > 8:
    print("\n>> This card is fp64-crippled (consumer die). float64 results below\n"
          ">> understate what SSJ does on a datacentre GPU. The 'mixed' rows are\n"
          ">> the ones that matter here; for a clean fp64 read, use an A100/V100.")
else:
    print("\n>> This card has real fp64 throughput. The float64 rows below are\n"
          ">> the meaningful comparison.")

In [ ]:
# =====================================================================
#  2 - SSJ, backend-agnostic (same code path on numpy and cupy arrays)
# =====================================================================
import numpy as np


def _am(A):
    """Array module for A, without importing cupy unless a cupy array shows up."""
    if type(A).__module__.partition(".")[0] == "cupy":
        import cupy
        return cupy
    return np


def _rdtype(A):
    return np.float32 if A.dtype in (np.float32, np.complex64) else np.float64


def off_frob(B):
    xp = _am(B)
    return float(xp.linalg.norm(B - xp.diag(xp.diag(B)), ord="fro"))


def _spec_norm(A, iters=100):
    """||A||_2 by power iteration on Hermitian A.

    A power-iteration estimate can only come in at or below the true norm,
    which makes the effective tolerance tighter, never looser.

    GPU deviation: the normalisation uses a 0-d device array instead of a
    Python float, so the loop costs one host sync at the end rather than 100.
    """
    xp = _am(A)
    v = 1.0 + 0.01 * xp.arange(A.shape[0], dtype=_rdtype(A))
    v = v / xp.linalg.norm(v)
    est = None
    for _ in range(iters):
        w = A @ v
        est = xp.linalg.norm(w)
        v = w / xp.where(est > 0, est, 1.0)
    return float(est)


def _angles(B):
    """Saturated Jacobi generator K_ij = 1/2 arctan(2|B_ij| / (d_j - d_i)).

    B must be exactly Hermitian on entry; then K is exactly anti-Hermitian.
    Zero gap -> +-pi/4 * phase(B_ij); B_ij = 0 -> 0.
    """
    xp = _am(B)
    n = B.shape[0]
    d = xp.real(xp.diag(B))
    absB = xp.abs(B)
    gap = d[None, :] - d[:, None]            # exactly antisymmetric
    with np.errstate(divide="ignore", invalid="ignore"):
        theta = 0.5 * xp.arctan(2.0 * absB / gap)
    theta = xp.nan_to_num(theta, nan=0.0)    # 0/0 at zero gap: no rotation
    # At a zero gap with B_ij != 0 the raw formula returns +pi/4 on BOTH (i,j)
    # and (j,i) -- each sees a +0 gap -- which would break anti-Hermiticity.
    # Orient the tie by the triangle instead. Applied unconditionally: the
    # `if tie.any()` guard of the CPU version would cost a host sync per sweep.
    tie = (gap == 0.0) & (absB > 0.0)
    one = xp.ones((n, n), dtype=theta.dtype)
    theta = xp.where(tie, (np.pi / 4.0) * (xp.triu(one, 1) - xp.tril(one, -1)),
                     theta)
    if B.dtype.kind == "c":
        nz = absB > 0
        phase = xp.where(nz, B / xp.where(nz, absB, 1.0), 0.0)
    else:
        phase = xp.sign(B)
    K = theta * phase
    xp.fill_diagonal(K, 0.0)
    return K


def _orth_qr(M):
    """Orthonormal factor of M, sign-fixed so diag(R) > 0."""
    xp = _am(M)
    Q, R = xp.linalg.qr(M)
    d = xp.diag(R).copy()
    if d.dtype.kind == "c":
        a = xp.abs(d)
        ph = xp.where(a > 0, d / xp.where(a > 0, a, 1.0), 1.0)
    else:
        ph = xp.where(xp.sign(d) == 0, 1.0, xp.sign(d))
    return Q * ph


def _orth_cholqr2(M):
    """Two rounds of G = M^H M, R = chol(G), M <- M R^-1.

    Safe here because sigma(X(I+K)) stays within a small factor of 1, so G is
    well-conditioned. Every operation is a gemm or a small triangular solve,
    which is why this is a GPU candidate even though it measured ~3x slower
    than Householder QR on CPU.
    """
    xp = _am(M)
    for _ in range(2):
        R = xp.linalg.cholesky(M.conj().T @ M).conj().T
        M = M @ xp.linalg.inv(R)
    return M


def _orth_ns(Y, target, max_iter=60):
    """Newton-Schulz until ||Y^H Y - I||_F < target. Requires sigma(Y) < sqrt(3).

    Each iteration's Y^H Y doubles as the error monitor, so the stopping test
    is free. Returns (Y, gemm_count).
    """
    xp = _am(Y)
    eye = xp.eye(Y.shape[0], dtype=Y.dtype)
    gemms, prev = 0, np.inf
    for _ in range(max_iter):
        G = Y.conj().T @ Y
        gemms += 1
        dev = float(xp.linalg.norm(G - eye, ord="fro"))
        if dev < target or dev > 0.9 * prev:      # converged, or stalled at roundoff
            break
        prev = dev
        Y = Y @ (1.5 * eye - 0.5 * G)
        gemms += 1
    return Y, gemms


def _pow_norm(K, v0=None, iters=25):
    """||K||_2 for anti-Hermitian K, by power iteration on K^H K.

    Returns (estimate, v) so the dominant vector warm-starts the next sweep --
    the generator changes slowly, so a warm estimate needs only a few
    iterations. A modest underestimate is safe: it weakens the cap slightly,
    and the sqrt(3) Newton-Schulz headroom absorbs it.

    GPU deviation: sync-free inner loop, as in _spec_norm.
    """
    xp = _am(K)
    if v0 is None:
        v = 1.0 + 0.01 * xp.arange(K.shape[0], dtype=_rdtype(K))
    else:
        v, iters = v0, 8
    v = v / xp.linalg.norm(v)
    est = None
    for _ in range(iters):
        w = K.conj().T @ (K @ v)
        est = xp.linalg.norm(w)
        v = w / xp.where(est > 0, est, 1.0)
    return float(xp.sqrt(est)), xp.where(est > 0, v, 1.0)


def ssj_eigh(A, tol=1e-13, method="gemm", max_sweeps=1000, ns_switch=0.5,
             gemm_cap=1.0, gemm_ns_factor=0.05, X0=None, precision="full",
             mixed_switch=1e-4, block_m=0, block_passes=2, block_fn=None,
             return_info=False):
    """Eigendecomposition of Hermitian A by Simultaneous Saturated Jacobi.

        B = X^H A X ;  K_ij = 1/2 atan(2 B_ij / (d_j - d_i)) ;  X <- orth(X(I+K))

    method   : "gemm" (factorization-free, all matmul -- the GPU choice),
               "qr", "auto" (QR then a Newton-Schulz endgame), "cholqr2".
    precision: "mixed" runs the linear phase in fp32 down to `mixed_switch`,
               then warm-starts an fp64 phase from that basis. Sound because
               the map is memoryless: every sweep re-derives its angles from a
               fresh B, so low-precision sweeps cannot poison the final
               accuracy, only the warm start's quality.
    block_m  : >0 enables the SSJ-BC block-cluster preconditioner from cell 3.
    """
    xp = _am(A)
    A = xp.asarray(A)

    if precision == "mixed":
        low = np.complex64 if A.dtype.kind == "c" else np.float32
        full = np.complex128 if A.dtype.kind == "c" else np.float64
        _, V_low, i_low = ssj_eigh(
            A.astype(low), tol=max(mixed_switch, tol), method=method,
            max_sweeps=min(max_sweeps, 100), ns_switch=ns_switch,
            gemm_cap=gemm_cap, gemm_ns_factor=gemm_ns_factor,
            X0=None if X0 is None else xp.asarray(X0).astype(low),
            block_m=block_m, block_passes=block_passes, block_fn=block_fn,
            return_info=True)
        w, V, info = ssj_eigh(
            A.astype(full), tol=tol, method=method, max_sweeps=max_sweeps,
            ns_switch=ns_switch, gemm_cap=gemm_cap,
            gemm_ns_factor=gemm_ns_factor, X0=V_low.astype(full),
            block_m=block_m, block_passes=block_passes, block_fn=block_fn,
            return_info=True)
        info["sweeps_low"] = i_low["sweeps"]
        info["sweeps_total"] = i_low["sweeps"] + info["sweeps"]
        info["gemms"] += i_low["gemms"]
        return (w, V, info) if return_info else (w, V)

    n = A.shape[0]
    norm_A = _spec_norm(A)
    X = xp.eye(n, dtype=A.dtype) if X0 is None else _orth_qr(
        xp.asarray(X0).astype(A.dtype, copy=False))
    eye = xp.eye(n, dtype=A.dtype)
    history, gemms, sweeps, converged, pv = [], 0, 0, False, None

    for _ in range(max_sweeps):
        B = X.conj().T @ (A @ X)
        B = (B + B.conj().T) / 2.0           # exact symmetry for the angle map
        rel_off = off_frob(B) / norm_A
        history.append(rel_off)
        if rel_off <= tol:
            converged = True
            break

        if block_m and block_fn is not None:
            for r in range(block_passes):
                B, X = block_fn(B, X, block_m,
                                0 if (sweeps + r) % 2 == 0 else block_m // 2)
            rel2 = off_frob(B) / norm_A
            if rel2 <= tol:
                converged = True
                history.append(rel2)
                sweeps += 1
                break

        K = _angles(B)
        Y = X @ (eye + K)                    # = X + X @ K

        if method == "qr":
            X = _orth_qr(Y)
        elif method in ("auto", "cholqr2"):
            if float(xp.linalg.norm(K, ord="fro")) < ns_switch:
                # Target the quadratic-tail scale rel_off^2, floored at a
                # fraction of tol: with the block pass the error can fall
                # several orders in ONE sweep, and then rel_off^2 is no longer
                # below the NEXT error, so the orthogonality defect would
                # survive into the answer.
                tgt = rel_off * rel_off
                if block_m:
                    tgt = min(tgt, 0.1 * tol)
                X, g = _orth_ns(Y, target=tgt)
                gemms += g
            else:
                X = _orth_qr(Y) if method == "auto" else _orth_cholqr2(Y)
        elif method == "gemm":
            gemms += 3                       # B (2) + X @ K (1)
            sigma, pv = _pow_norm(K, v0=pv)
            if sigma > gemm_cap:
                K = K * (gemm_cap / sigma)
                Y = X @ (eye + K)
                gemms += 1
            # The polar factor is invariant under positive scaling, and
            # sigma(I+K) spans [1, sqrt(1+s^2)] exactly for anti-Hermitian K,
            # so dividing by the geometric mean centres the singular values
            # around 1 and saves Newton-Schulz iterations.
            s = min(sigma, gemm_cap)
            if s > 0.1:
                Y = Y / (1.0 + s * s) ** 0.25
            tgt = gemm_ns_factor * rel_off
            if block_m:
                tgt = min(tgt, 0.1 * tol)
            X, g = _orth_ns(Y, target=tgt)
            gemms += g
        else:
            raise ValueError(f"unknown method {method!r}")
        sweeps += 1
    else:
        B = X.conj().T @ (A @ X)
        B = (B + B.conj().T) / 2.0
        history.append(off_frob(B) / norm_A)

    w = xp.real(xp.diag(B))
    order = np.argsort(w, kind="stable") if xp is np else xp.argsort(w)
    w, V = w[order], X[:, order]
    info = {"sweeps": sweeps, "gemms": gemms, "history": history,
            "converged": converged, "norm_A": norm_A}
    return (w, V, info) if return_info else (w, V)


print("SSJ loaded.")

In [ ]:
# =====================================================================
#  3 - SSJ-BC: the block-cluster preconditioner, batched for the GPU
# =====================================================================
# Why it exists: diag(B) starts at spread ||A||/sqrt(n) and must climb to the
# true spectral spread, which costs ~(1/2)log n sweeps. While the spread is
# small nearly every gap is comparable to every coupling, so thousands of
# pairs saturate at +-pi/4 and the contraction rate sits at 0.99. Diagonalising
# sorted-contiguous blocks exactly hands the iteration ~sqrt(m) of that spread
# for free. On CPU this took 20/24/29 sweeps down to 9/11/14 at m=32.
#
# Correctness: for block-diagonal orthogonal P, the (I,J) block of P^T B P is
# Q_I^T B_IJ Q_J, whose Frobenius norm equals ||B_IJ||_F. Off-block masses are
# preserved individually and within-block off-diagonal mass is zeroed, so a
# block pass is a strict monotone reduction of off(B) for ANY grouping.
#
# GPU shaping: the CPU version loops over blocks in Python, issuing hundreds of
# tiny kernels -- latency-bound. Here the sorted diagonal is rolled cyclically
# by `offset` instead of being cut into a ragged head and tail, so every block
# has size exactly m and ONE batched eigh handles all of them. The grouping
# that changes (extreme-low with extreme-high entries land together) is
# harmless by the paragraph above.

def block_pass(B, X, m, offset):
    """One exact block-Jacobi pass on sorted-contiguous blocks of size m."""
    xp = _am(B)
    n = B.shape[0]
    nb = n // m
    if nb < 2:
        return B, X
    keep = nb * m

    p = xp.argsort(xp.real(xp.diag(B)))
    if offset:
        p = xp.roll(p, -int(offset))
    B = B[p][:, p]
    X = X[:, p]

    sub = B[:keep, :keep].reshape(nb, m, nb, m)
    blocks = sub[xp.arange(nb), :, xp.arange(nb), :]          # (nb, m, m)
    blocks = (blocks + blocks.conj().transpose(0, 2, 1)) / 2.0
    Q = xp.linalg.eigh(blocks)[1]                             # batched

    # Block-diagonal Q as one (keep, keep) operator: applying it is then two
    # ordinary gemms rather than 2*nb small ones.
    Qfull = xp.zeros((keep, keep), dtype=B.dtype)
    Qfull.reshape(nb, m, nb, m)[xp.arange(nb), :, xp.arange(nb), :] = Q

    X[:, :keep] = X[:, :keep] @ Qfull
    B[:, :keep] = B[:, :keep] @ Qfull
    B[:keep, :] = Qfull.conj().T @ B[:keep, :]
    return (B + B.conj().T) / 2.0, X


print("SSJ-BC block pass loaded (batched).")

In [ ]:
# =====================================================================
#  4 - Timing harness and problem generators
# =====================================================================
import time


def sync(xp):
    if xp is not np:
        xp.cuda.Device().synchronize()


def timed(fn, xp, reps=3):
    """Min-of-reps wall time. The synchronize() calls are not optional: without
    them you measure kernel LAUNCH time, which for a fast GPU op is ~10 us
    regardless of the work queued behind it."""
    fn()                                    # warm up: autotune, workspaces, JIT
    sync(xp)
    best = float("inf")
    for _ in range(reps):
        t0 = time.perf_counter()
        fn()
        sync(xp)
        best = min(best, time.perf_counter() - t0)
    return best


def goe(n, xp, seed=0, dtype=None):
    """GOE: flat spectrum, the hardest case for SSJ (no graded structure to
    exploit, so the prologue lever does nothing)."""
    rng = xp.random.default_rng(seed)
    M = rng.standard_normal((n, n))
    A = (M + M.T) / xp.sqrt(xp.asarray(2.0 * n))
    return A.astype(dtype or np.float64)


def check(A, w, V, xp):
    """Residual and orthogonality, both relative to ||A||_F.

    A fast wrong answer is still wrong, so every timed configuration below
    reports these alongside its wall time.
    """
    resid = float(xp.linalg.norm(A @ V - V * w, ord="fro")) / float(
        xp.linalg.norm(A, ord="fro"))
    ortho = float(xp.linalg.norm(
        V.conj().T @ V - xp.eye(A.shape[0], dtype=V.dtype), ord="fro"))
    return resid, ortho


print("harness loaded.")

In [ ]:
# =====================================================================
#  5 - The GPU sweep: SSJ configurations vs cuSOLVER syevd
# =====================================================================
SIZES = [512, 1024, 2048]        # add 4096 if the card has memory and patience

# Labels are shared with the CPU control in cell 6 so the two tables can be
# joined on (n, label) without any string surgery.
BASE = "gemm f64"
BASE_BC = "gemm f64 +BC32"
CONFIGS = [
    (BASE,             dict(method="gemm")),
    ("cholqr2 f64",    dict(method="cholqr2")),
    ("gemm mixed",     dict(method="gemm", precision="mixed")),
    (BASE_BC,          dict(method="gemm", block_m=32, block_fn=block_pass)),
    ("gemm mixed +BC32", dict(method="gemm", precision="mixed",
                             block_m=32, block_fn=block_pass)),
]

gpu_ratio, gpu_rows = {}, []
for n in SIZES:
    A = goe(n, cp, seed=1)
    t_gemm = timed(lambda: A @ A, cp, reps=5)
    t_ref = timed(lambda: cp.linalg.eigh(A), cp, reps=3)
    print(f"\n=== n={n} ===")
    print(f"  fp64 gemm       {t_gemm*1e3:9.2f} ms")
    print(f"  cuSOLVER eigh   {t_ref*1e3:9.2f} ms  "
          f"= {t_ref/t_gemm:6.1f} gemm-equivalents"
          f"   (SSJ needs ~55-80)")
    print(f"  {'config':>18}{'ms':>10}{'vs cuSOLVER':>13}"
          f"{'sweeps':>8}{'resid':>10}{'ortho':>10}")

    for label, kw in CONFIGS:
        try:
            t = timed(lambda: ssj_eigh(A, **kw), cp, reps=2)
            w, V, info = ssj_eigh(A, return_info=True, **kw)
            sync(cp)
            resid, ortho = check(A, w, V, cp)
            sw = info.get("sweeps_total", info["sweeps"])
            flag = "" if info["converged"] else "  NOT CONVERGED"
            print(f"  {label:>18}{t*1e3:10.1f}{t/t_ref:12.1f}x{sw:8d}"
                  f"{resid:10.1e}{ortho:10.1e}{flag}")
            gpu_ratio[(n, label)] = t / t_ref
            gpu_rows.append((n, label, t, t_ref, t / t_ref, sw, resid, ortho))
        except Exception as e:
            print(f"  {label:>18}  FAILED: {type(e).__name__}: {e}")

print("\nDone. `gpu_rows` / `gpu_ratio` hold the raw numbers.")

In [ ]:
# =====================================================================
#  6 - CPU control: the same code, so the answer is a SHIFT in the ratio
# =====================================================================
# Absolute times on a Colab vCPU are meaningless. The ratio SSJ/LAPACK is not:
# it is the same quantity measured on the GPU against cuSOLVER, so the
# difference between the two ratios is the actual result of this notebook.

CPU_SIZES = [512, 1024]          # the vCPU is slow; keep this modest

gpu_ratio = globals().get("gpu_ratio", {})   # so this cell stands alone too
cpu_ratio = {}
print(f"  {'n':>6}{'config':>18}{'SSJ ms':>10}{'LAPACK ms':>12}{'ratio':>9}"
      f"{'sweeps':>8}{'resid':>10}")
for n in CPU_SIZES:
    A = goe(n, np, seed=1)
    t_ref = timed(lambda: np.linalg.eigh(A), np, reps=3)
    for label, kw in [(BASE, dict(method="gemm")),
                      (BASE_BC, dict(method="gemm", block_m=32,
                                     block_fn=block_pass))]:
        t = timed(lambda: ssj_eigh(A, **kw), np, reps=2)
        w, V, info = ssj_eigh(A, return_info=True, **kw)
        resid, _ = check(A, w, V, np)
        cpu_ratio[(n, label)] = t / t_ref
        print(f"  {n:6d}{label:>18}{t*1e3:10.1f}{t_ref*1e3:12.1f}"
              f"{t/t_ref:8.1f}x{info['sweeps']:8d}{resid:10.1e}")

# ------------------------------------------------------------------ verdict
print("\n" + "=" * 68)
print("  THE NUMBER THIS NOTEBOOK EXISTS TO PRODUCE")
print("=" * 68)
print(f"  {'n':>6}{'config':>18}{'CPU ratio':>12}{'GPU ratio':>12}{'shift':>10}")
shown = 0
for (n, label), rc in sorted(cpu_ratio.items()):
    rg = gpu_ratio.get((n, label))
    if rg is None:
        continue
    shown += 1
    print(f"  {n:6d}{label:>18}{rc:11.1f}x{rg:11.1f}x{rc/rg:9.2f}x")
if not shown:
    print("  (no matching GPU rows -- run cell 5 first)")

print("""
  shift = (SSJ/LAPACK on CPU) / (SSJ/cuSOLVER on GPU)

    > 1   the GPU narrows SSJ's disadvantage by that factor
    ~ 1   SSJ's all-gemm structure buys nothing here; the loss is flop count
    < 1   cuSOLVER exploits this GPU better than SSJ does

  A shift near 1 measured on an fp64-crippled card (see cell 1) is NOT
  conclusive -- rerun on an A100 or V100 before drawing any conclusion. A
  shift near 1 on a card with real fp64 throughput IS conclusive, and is a
  genuine negative result worth recording as one.""")

In [ ]:
# =====================================================================
#  7 - Warm-started tracking: the regime where SSJ can actually WIN
# =====================================================================
# A cold full solve is the HARDEST case for SSJ. The favourable case is
# re-solving a matrix that moved slightly -- molecular dynamics, parameter
# sweeps, continuation, self-consistent field loops. SSJ accepts the previous
# eigenbasis as X0 and lands directly in the quadratic tail (1-5 sweeps
# instead of 20-29). syevd has no way to accept a warm start: it pays full
# price every single time.
#
# A ratio BELOW 1.0 in this table is SSJ beating cuSOLVER outright.

TRACK_SIZES = [1024, 2048]
EPSILONS = [1e-2, 1e-4, 1e-6]

print(f"  {'n':>6}{'eps':>8}{'SSJ warm ms':>13}{'cuSOLVER ms':>13}"
      f"{'ratio':>9}{'sweeps':>8}{'resid':>10}")
for n in TRACK_SIZES:
    A = goe(n, cp, seed=1)
    _, V0 = cp.linalg.eigh(A)               # the "previous" eigenbasis
    rng = cp.random.default_rng(5)
    P = rng.standard_normal((n, n))
    P = (P + P.T) / 2.0
    P = P / float(cp.linalg.norm(P, ord="fro"))

    for eps in EPSILONS:
        A2 = A + eps * P
        t_ref = timed(lambda: cp.linalg.eigh(A2), cp, reps=2)
        t_warm = timed(lambda: ssj_eigh(A2, X0=V0, method="gemm"), cp, reps=2)
        w, V, info = ssj_eigh(A2, X0=V0, method="gemm", return_info=True)
        sync(cp)
        resid, _ = check(A2, w, V, cp)
        mark = "   <-- SSJ WINS" if t_warm < t_ref else ""
        print(f"  {n:6d}{eps:8.0e}{t_warm*1e3:13.1f}{t_ref*1e3:13.1f}"
              f"{t_warm/t_ref:8.2f}x{info['sweeps']:8d}{resid:10.1e}{mark}")

print("""
If the ratio is below 1 here but above 1 in cell 5, the honest summary is:
"SSJ is not a general-purpose replacement for syevd, but it is the faster
option for tracking a slowly-varying matrix." That is a narrower claim than
the campaign started with, and a defensible one.""")

## What the possible outcomes mean

SSJ costs ~55–80 gemm-equivalents; `dsyevd` costs 8–18 on CPU. For SSJ to win
anywhere, the incumbent's cost in *the same unit* has to rise. Cell 5 prints
exactly that for cuSOLVER ("= N gemm-equivalents"), so you can read the
outcome off that line before even looking at SSJ:

- **cuSOLVER above ~60 gemm-equivalents** — the tridiagonal reduction is
  hurting on this hardware and SSJ is genuinely in range. This is the case the
  method was designed for.
- **cuSOLVER at 15–30** — the gap narrowed but SSJ still loses on flops. The
  interesting configurations are then the ones that cut flops rather than
  reshape them: `mixed`, `+BC32`, and warm starts.
- **cuSOLVER still under ~15** — the all-gemm argument does not pay off here,
  and that is a real finding.

## Reading cell 5 and cell 7 together

These two cells ask different questions and can easily disagree:

- **Cell 5 (cold solve)** is the hardest case for SSJ and the one it is most
  likely to lose. A loss here is expected, not disqualifying.
- **Cell 7 (warm tracking)** is the case SSJ is structurally suited to, and
  the only one where a ratio below 1 is plausible.

If cell 7 shows a win and cell 5 does not, the defensible claim is narrow and
specific: *SSJ is not a general-purpose `syevd` replacement, but it is the
faster option for tracking a slowly-varying matrix.* Resist widening it.

One lever this notebook does not isolate: **mixed precision on tensor cores.**
The fp32 phase does most of the sweeps, and tensor cores run it far faster
than the fp64 units, so the CPU-measured 1.3–1.4× is a floor rather than an
estimate. The `mixed` rows in cell 5 include it, but conflated with everything
else.

## Known weaknesses of this port, stated plainly

- `_orth_ns` syncs once per Newton–Schulz iteration to test its stopping
  criterion (typically 2–5 per sweep). A fixed iteration count would remove
  them at the cost of doing unnecessary work.
- The block pass builds a dense `(keep, keep)` block-diagonal `Qfull` so the
  application is two gemms. That wastes `O(n²)` memory traffic on zeros;
  a batched `matmul` over `(nb, m, m)` views trades that for `nb` small
  kernels. Which wins is a size-dependent question this notebook does not
  settle.
- Sweep counts are hardware-independent, so they are directly comparable to
  the CPU numbers quoted above. Wall times are not comparable across
  machines — only ratios are.